In [ ]:
import sys
from pathlib import Path

import polars as pl

OUT = Path(r"D:\MIMIC\trajectory_out")


In [ ]:
df_episode = pl.read_parquet(OUT / "episodes.parquet")
df_event = pl.scan_parquet(OUT / "events.parquet")


In [ ]:
df_event.collect_schema()

In [ ]:
df_event.head(1000).collect()["text"].unique()


# all patients has info of both visit and events 

In [ ]:
patient_w_event = set(df_event.select("subject_id").unique().collect(engine="streaming")["subject_id"].to_list())
patient_w_eps = set(df_episode.select("subject_id").unique()["subject_id"].to_list())


In [ ]:
print(len(patient_w_event - patient_w_eps))
print(len(patient_w_eps - patient_w_event))
print(len(df_episode.filter(pl.col("start_days_since_birth") > pl.col("end_days_since_birth"))))

print(len(df_episode.filter(pl.col("start_days_since_birth") < 0 )))
print(len(df_episode.filter(pl.col("end_days_since_birth") < 0 )))


In [ ]:
(df_event.select("source", "time_of_day")
 .collect(streaming = True)
 .filter(pl.col("time_of_day").is_null())
 .group_by("source")
 .agg(pl.len())
)

In [ ]:
df_episode.select("episode_id", "kind").group_by("kind").agg(pl.len())

In [ ]:
import polars as pl

e = pl.scan_parquet(r"D:\MIMIC\trajectory_out\events.parquet")
print(e.group_by("setting").len().collect(engine="streaming"))                      # expect a row for 'outpatient'
print(e.group_by("source").agg(
    pl.col("hadm_id").is_null().sum().alias("rows_without_hadm"))
    .collect(engine="streaming"))

ep = pl.read_parquet(r"D:\MIMIC\trajectory_out\episodes.parquet")
print(ep.group_by("kind").len())                        # expect 'outpatient_day'

In [ ]:
import polars as pl

e = pl.scan_parquet(r"D:\MIMIC\trajectory_out\events.parquet")
print(e.group_by(["source", "time_basis"]).len().sort("source").collect(engine="streaming"))